# October 12 full day — without dev sell all

**October 12, 2026, 00:00–24:00 UTC launches; 10,000 SOL starting wallet.**
The developer-sold-all entry requirement is disabled. Original sizing, the first
two-slot entry window, one-slot execution delay, +5% own take profit, five-minute
fallback and 5% slippage remain unchanged. Creator fees and reserve-capped exits
use the saved native backtest implementation.

**Recorded outcome:** 251 profitable tokens, 318 losing, 80 unresolved and 238
submitted failed entries. Realized net is **−109.602394368 SOL**, including failed
entry fees. **152.319150420 SOL** remains committed to unresolved exposure.
Final portfolio PnL is unknown; unresolved tokens are not counted as realized losses.

This notebook reads frozen local results and chart context. It does not query a
server or run another backtest. Select **Python 3.13.12 (Sniper research)** to rerun;
executed tables and example charts are already saved.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.io as pio
from IPython.display import display, Markdown
ROOT = next((candidate for p in [Path.cwd(), *Path.cwd().parents]
             for candidate in (p, p/'onchain-backtest-engine')
             if (candidate/'src/backtest').is_dir()), None)
assert ROOT is not None, 'Open the notebook inside the onchain-backtest-engine workspace'
sys.path.insert(0, str(ROOT/'notebooks'))
from sniper_full_day_review import FullDayReview, LABEL
pio.renderers.default = 'plotly_mimetype'
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 90)
review = FullDayReview()
display(review.overview())

,period,wallet_SOL,signals,buys,profitable,losing,unresolved,failed_entries,risk_rejected,realized_net_SOL,unresolved_cost_SOL,final_portfolio_PnL
0,"October 12, 2026 — full UTC day",10000,887,649,251,318,80,238,0,-109.602394368,152.319150420,Unknown: unresolved positions are not marked


## Strategy settings and accounting

Realized PnL records completed positions at their sell fill, including their
failed sell fees, plus fees for submitted failed entries. Cash change is separate:
it also includes capital spent on open tokens and posted rent. Open committed
cost includes those positions' entry costs and failed execution fees.

Buys and sells use native fills and settlement ledger amounts. Green ▲ and red ▼
mark average **curve execution prices**; external fees are separate. Hollow diamonds
are recorded decisions, shown at historical reference prices. Black × marks an
attempt that failed without a fill. Landing at the start of the next slot is a
simulation assumption, not an observed live order.

In [2]:
display(pd.Series(review.summary['policy'], name='Saved native policy').to_frame())

,Saved native policy
early_slots,2
minimum_sniper_buy_lamports,1000000000
target_return_bps,500
epsilon_lamports,1000000
entry_delay_slots,1
exit_delay_slots,1
holding_seconds,300
entry_network_fee_lamports,105000
exit_network_fee_lamports,105000
entry_tip_lamports,0


## PnL curve and capital exposure

The first panel is realized net PnL. The final point must equal the saved native
result. The second panel is cash change, and the third is open committed cost;
neither is a marked portfolio valuation.

In [3]:
display(review.strategy_figure())

## Results by outcome and exit reason

Closed-position PnL below excludes failed entries; the overview and realized
curve include their fees. Submitted failures and risk rejections are distinct
from losing positions.

In [4]:
display(review.table.groupby('group', dropna=False).agg(
    tokens=('mint','size'), closed_PnL_SOL=('closed_PnL_SOL', lambda x:x.sum(min_count=1)),
    failed_fees_SOL=('failed_fees_SOL','sum'), unresolved_cost_SOL=('unresolved_cost_SOL','sum')))
display(review.table[review.table.group.isin(['Profitable','Losing'])].groupby('reason').agg(
    tokens=('mint','size'), profitable=('closed_PnL_SOL', lambda x:int((x>0).sum())),
    closed_PnL_SOL=('closed_PnL_SOL','sum')))

,tokens,closed_PnL_SOL,failed_fees_SOL,unresolved_cost_SOL
group,,,,
Entry failed,238,NaN,0.024990,0.00000
Losing,318,-173.715062,0.000210,0.00000
Profitable,251,64.137658,0.001260,0.00000
Unresolved,80,NaN,0.000525,152.31915


,tokens,profitable,closed_PnL_SOL
reason,,,
TARGET,248,248,64.023214
TIMEOUT,321,3,-173.600618


## Select any token

Filter by outcome and search by symbol or mint. Each selection shows the entry
and exit, tracked buyer, developer, fees, signal thresholds, complete recorded
order lifecycle and any conditional replay blockers. The selector requires an
active kernel. Saved examples follow below for viewing without one.

Wallet inventory and positive-wallet counts are modeled from retained swaps and
known transfers. They are **not exact holders**. The chart stops inventory at its
first replay blocker. Historical prices may continue after modeled coverage ends.

In [5]:
display(review.browser())

## All entries and attempts

These are all 887 signals. Scroll horizontally for entry/exit slots, the tracked
wallet, developer and GMGN link. Full tables remain available as CSV.

In [6]:
review.display_table()

mint,token,group,reason,first_causal_blocker,execution_failure_reason,entry_SOL,closed_PnL_SOL,failed_fees_SOL,unresolved_cost_SOL,signal_slot,entry_slot,exit_slot,tracked_wallet,developer,gmgn_url,detailed_trace_available
A2vTTZyjaTn2Ly3FTnUqi34rsDptNzA968owwFbrpump,MIBI,Profitable,TARGET,,,2.828017490,0.433451085,0.000000000,0.000000000,455841257,455841258,455841342,D5BDDkBtjjva35nwu45WfmZgytsog9GST2tHQaZUYuFX,9SuiFEj4gRbdu7F8ikgrAxWMNt9XEbLRWiEYo2Hk2C5u,GMGN,False
7bHUu6Lxcy75vuR6EgNuepjwpUUBthDbwVqAode2pump,ARCDOG,Entry failed,RESERVE_DISCONTINUITY,,MINIMUM_OUTPUT_NOT_MET,0.000000000,—,0.000105000,0.000000000,455841299,<NA>,<NA>,DHrKwYTP7aKaiQ8rjBBzbB4Zo2xV1LpbKYE8dsFQZEid,9aHdkTFrbYHB4MZhUU64sUSh1furcLtvwPBGG8Z3WTXz,GMGN,False
2ikMciSZ1nK1X9E8X7U5trw3Tc9jW1Ud7FjrhAxPpump,apelink,Entry failed,MINIMUM_OUTPUT_NOT_MET,,MINIMUM_OUTPUT_NOT_MET,0.000000000,—,0.000105000,0.000000000,455841406,<NA>,<NA>,9HzDhrKAR5Fp6LBmTsNhrUAUC4ZixpYv5Vn1ToA5yTx5,9fJVBF4WaFodQo5BHHHoo9rm6HuJg4P8KRQM23bKHPoa,GMGN,False
BYgPwdCQ4xEQrRuEud2uFU2AYPCdYGoNbu6yJr2hpump,Nuts,Profitable,TARGET,,,1.328059060,0.767069205,0.000105000,0.000000000,455842072,455842073,455842077,9qz7cWBqYBP2TnGfzMmxX64G4PQip7HtogjPBBnMhWpw,D8Sw1gfWfcYJB2fQnPMeQ7jYzT7hfnz32N3yjbM6FkUy,GMGN,False
DraQMm86aowauGjQ1HTcZUkuh7JQfn89qyn3sdgTpump,Jeremiah,Profitable,TARGET,,,1.422415220,0.318340930,0.000000000,0.000000000,455842135,455842136,455842137,BXB7FF4m5x74ZoUvVHfPJnVBbqWHpMEU1WNUZAL9W8PL,BifnsfduLvjEfQXJssiDZMuT5TW8jFiDyrD1jmnY7LZe,GMGN,False
HPBYBXbKucX3aE4kQ1WucumFGZmEnVLBuevQAbuupump,muffincat,Entry failed,MINIMUM_OUTPUT_NOT_MET,,MINIMUM_OUTPUT_NOT_MET,0.000000000,—,0.000105000,0.000000000,455842354,<NA>,<NA>,9qz7cWBqYBP2TnGfzMmxX64G4PQip7HtogjPBBnMhWpw,FePSGRavtQty99PFUty5WnaK6M7eibcEEpZRcrmZ6Xi1,GMGN,False
3Ztm75oVzHKkXsCNx8UmFnKkhMThf7yBpY9oTspKLKbv,ENTH,Losing,TIMEOUT,,,18.243462859,-2.170983484,0.000000000,0.000000000,455842658,455842659,455844037,2ZAD7NzumGLMJm6yqffZKzy9LUwqBRgwkL4CPt6Sz8Yq,CCxZZjA6r7T1YNndsEgdXaRcL8SMZzU9aJ3j56xMvKMb,GMGN,False
CzVTWQgbjBMQhGrUsAz15hX8qdpFxTWTGeatbR5Bpump,VAKAFU,Losing,TIMEOUT,,,4.128785331,-0.188560373,0.000000000,0.000000000,455842691,455842692,455844071,EaoBCDSwUfzyqthe55Qne8StLxS64TwSq7m5wD2umsFN,6Xrus3qLHYZBh5MPHFwLuoC6YKwzhLZwDePfQbqhzjKL,GMGN,False
4A6FnkFxfLUA7rGwVkYKXyenXsn7Zhbw8NuXVqU8pump,S&P500,Profitable,TARGET,,,1.782178949,0.100335405,0.000000000,0.000000000,455842766,455842767,455842770,2QVCJaDqok4PFsSYAS2hdgooBJeVu6X5Y91gqPkXEKft,DQbQLJGbifqREhoMZu111M4wTWitcPQ3nH7YkvvnHqFC,GMGN,False
FFnvBUT8ijcLhLML9mfWxQyrLc1ZtA1j8aKGU8bWpump,COMPANY,Entry failed,MINIMUM_OUTPUT_NOT_MET,,MINIMUM_OUTPUT_NOT_MET,0.000000000,—,0.000105000,0.000000000,455842855,<NA>,<NA>,CWiNFAEk4M1y6SjvnZkmxWDmhqSmnKFcnop1WoDW3S2J,CDVHXP2QAKDcwfa1CKvPg1E3hVwrHVw6JDcfXmShXvta,GMGN,False


## Profitable tokens — 251

In [7]:
review.display_table('Profitable')

mint,token,group,reason,first_causal_blocker,execution_failure_reason,entry_SOL,closed_PnL_SOL,failed_fees_SOL,unresolved_cost_SOL,signal_slot,entry_slot,exit_slot,tracked_wallet,developer,gmgn_url,detailed_trace_available
A2vTTZyjaTn2Ly3FTnUqi34rsDptNzA968owwFbrpump,MIBI,Profitable,TARGET,,,2.828017490,0.433451085,0.000000000,0.000000000,455841257,455841258,455841342,D5BDDkBtjjva35nwu45WfmZgytsog9GST2tHQaZUYuFX,9SuiFEj4gRbdu7F8ikgrAxWMNt9XEbLRWiEYo2Hk2C5u,GMGN,False
BYgPwdCQ4xEQrRuEud2uFU2AYPCdYGoNbu6yJr2hpump,Nuts,Profitable,TARGET,,,1.328059060,0.767069205,0.000105000,0.000000000,455842072,455842073,455842077,9qz7cWBqYBP2TnGfzMmxX64G4PQip7HtogjPBBnMhWpw,D8Sw1gfWfcYJB2fQnPMeQ7jYzT7hfnz32N3yjbM6FkUy,GMGN,False
DraQMm86aowauGjQ1HTcZUkuh7JQfn89qyn3sdgTpump,Jeremiah,Profitable,TARGET,,,1.422415220,0.318340930,0.000000000,0.000000000,455842135,455842136,455842137,BXB7FF4m5x74ZoUvVHfPJnVBbqWHpMEU1WNUZAL9W8PL,BifnsfduLvjEfQXJssiDZMuT5TW8jFiDyrD1jmnY7LZe,GMGN,False
4A6FnkFxfLUA7rGwVkYKXyenXsn7Zhbw8NuXVqU8pump,S&P500,Profitable,TARGET,,,1.782178949,0.100335405,0.000000000,0.000000000,455842766,455842767,455842770,2QVCJaDqok4PFsSYAS2hdgooBJeVu6X5Y91gqPkXEKft,DQbQLJGbifqREhoMZu111M4wTWitcPQ3nH7YkvvnHqFC,GMGN,False
2JJNpAczhb6tfus32C91eNPf1S6FBEDVaDfxdmvHpump,VMO,Profitable,TARGET,,,0.740063180,0.076158584,0.000000000,0.000000000,455847350,455847351,455847352,EaoBCDSwUfzyqthe55Qne8StLxS64TwSq7m5wD2umsFN,Cvn77ZsHcHGaya9YySFo1JWTSdHbTHrLMKS8CD1kCioZ,GMGN,False
7dEbQAAh3S2R35T8ogFmQv52eE5G6Dur8hQWqJKNpump,‎,Profitable,TIMEOUT,,,2.423446245,0.076200041,0.000000000,0.000000000,455849379,455849380,455850768,G9sQPg9Eimkwe72w1xH1tPUc9RFNuK6Ufw5AyWy6j7ob,BHphEMwvHW5euU8ciFQkAzYgpkdZaygcSPMyzXbod6L5,GMGN,False
B6qvFCbGuvha1UPcauooG4Ba8p7Aj4abtjduX8SHpump,LACY,Profitable,TARGET,,,1.558277166,0.525046933,0.000000000,0.000000000,455859441,455859442,455859447,E4mmLv7MugAQoJvqbzVSRTzn6LRWtsAQgX5dYze7WzCy,9mB1kqCLfb4CvG8sWihkJBhZ7WREKQ9VdjFGvDmwJBgy,GMGN,False
98aZvbY1X7ZSDUds7Kgb2eUqXjbtCGGowbqjK6UWpump,Bino,Profitable,TARGET,,,1.766727557,0.273882110,0.000000000,0.000000000,455859520,455859521,455859527,2mqrindMAjJEQPLhroYWyiYPo5h9iAsahfdd4QtsjwdY,2o638VrhywYQAtxrZ4WxjJvGLYc8WWjebATFuA9zJWZM,GMGN,False
364BmcVsmFMEM2nWqLY2wcFzayWcxTVZ4JQ9Rm97pump,PENUS,Profitable,TARGET,,,4.733892769,0.366954446,0.000000000,0.000000000,455860895,455860896,455860898,AEgTuy8Lg4wfnjm285mk3P2k1p39ofqNXk8b5VtYhdCs,2wwWQaQgWsJA5Afv2h4tHsthAUq89C71hrvefMRjkpG8,GMGN,False
613F4Vp5eDxxHUjuqF3scowYdro8fjTyZe48v4FApump,Diggs,Profitable,TARGET,,,1.274801920,0.575241764,0.000000000,0.000000000,455861831,455861832,455861836,4BHBSZA96XCaepYLvmteV6wLqKTqJPyfkME941X5PqHP,6mEtvQYVDFjSYg1cw33c4jQDmZZGaczhfC21xMXzNwhX,GMGN,False


## Losing tokens — 318

In [8]:
review.display_table('Losing')

mint,token,group,reason,first_causal_blocker,execution_failure_reason,entry_SOL,closed_PnL_SOL,failed_fees_SOL,unresolved_cost_SOL,signal_slot,entry_slot,exit_slot,tracked_wallet,developer,gmgn_url,detailed_trace_available
3Ztm75oVzHKkXsCNx8UmFnKkhMThf7yBpY9oTspKLKbv,ENTH,Losing,TIMEOUT,,,18.243462859,-2.170983484,0.000000000,0.000000000,455842658,455842659,455844037,2ZAD7NzumGLMJm6yqffZKzy9LUwqBRgwkL4CPt6Sz8Yq,CCxZZjA6r7T1YNndsEgdXaRcL8SMZzU9aJ3j56xMvKMb,GMGN,False
CzVTWQgbjBMQhGrUsAz15hX8qdpFxTWTGeatbR5Bpump,VAKAFU,Losing,TIMEOUT,,,4.128785331,-0.188560373,0.000000000,0.000000000,455842691,455842692,455844071,EaoBCDSwUfzyqthe55Qne8StLxS64TwSq7m5wD2umsFN,6Xrus3qLHYZBh5MPHFwLuoC6YKwzhLZwDePfQbqhzjKL,GMGN,False
78nYgkFzj4uaAgEkkcMjYj8P91k2Wyj2ou5miWBBpump,RAYLS,Losing,TIMEOUT,,,4.673902373,-0.564938177,0.000000000,0.000000000,455842931,455842932,455844304,DxhpC9c4kGkYJR34xUrfFeDx9vwb6yoHUMWrTCZybwtH,8dx4RHfjJhskX2aY2GJ1aJPLGRpJtW13FjRcknVdExSo,GMGN,True
AwaTyYehCQDErcKcznCmAP9VcjsHgwKgMX7tmvGcpump,Cases,Losing,TIMEOUT,,,1.674826280,-0.591664455,0.000000000,0.000000000,455843561,455843562,455844936,CwiJ7C75m1wb1jPZwRVZ36KfvRucxTwDiUZ1BYFyBU29,6vtufvJNZKYYNdUne3a9DGkKoXzNgUZf9k6pm7zPZnRg,GMGN,False
2tQzVURqQkMEGpHHqdFqHonvG4u3p8wyoGVg7Ls6pump,SELU,Losing,TIMEOUT,,,4.092158155,-0.312374029,0.000000000,0.000000000,455844363,455844364,455845739,AEgTuy8Lg4wfnjm285mk3P2k1p39ofqNXk8b5VtYhdCs,8CyLPk2Xzsrbakq7HdXArWFNQ1rKdCH2enxRmBPmw6mH,GMGN,False
6HH1Wow3SUZQ2ugjcBxDdjfy9VKhPExtNtEMsNNzpump,MINTY,Losing,TIMEOUT,,,1.085985600,-0.118997972,0.000000000,0.000000000,455846666,455846667,455848036,9ikSKkTCz6QWQKnwKChpefNkw9tAyzvj7XeDeF6UX9Pe,DicVVXTgPUi3FU5FcJU2KMG7XNMp5SSXAAJ1zgEUkQWy,GMGN,True
GdD2dv1jTLX7E6dqDmAi2uVXRDuhm9gmkVWYhLE5pump,ZAVE,Losing,TIMEOUT,,,0.955490140,-0.106305519,0.000000000,0.000000000,455847426,455847427,455848795,DxhpC9c4kGkYJR34xUrfFeDx9vwb6yoHUMWrTCZybwtH,AJR4Z4x4LP5271YYwimER7r39EnyCRQpdckDULwQdPra,GMGN,False
EkX23iEtz3EeB4okuK4M5sFH8s7SmaGjZUVnRebcpump,BOOGIE,Losing,TIMEOUT,,,1.739197000,-0.269039700,0.000000000,0.000000000,455848580,455848581,455849955,G9sQPg9Eimkwe72w1xH1tPUc9RFNuK6Ufw5AyWy6j7ob,9AvR4WS7wZLoeYAdNJbxLaQ4w5U8fFZ7FYDpDMddmbkv,GMGN,False
6hanGNNiPTmFiMBHt81uToeRLHL5HK3CmbLmq7Xopump,LUTIME,Losing,TIMEOUT,,,1.081371080,-0.079296259,0.000000000,0.000000000,455848617,455848618,455849994,AEgTuy8Lg4wfnjm285mk3P2k1p39ofqNXk8b5VtYhdCs,4yrPeq2cdjKA11UrujgcgCLSyhwBTYr7kM1v7e7eCqVR,GMGN,False
GhmC3F4we7x1Lo5tpnnXF1srsh8J5xy2gRy8NzPRpump,kidlon,Losing,TIMEOUT,,,1.736019046,-0.877084978,0.000000000,0.000000000,455849282,455849283,455850665,CD5gw9pjrX2oau7dCgWyMTnpVBDhB2b5CwRnJqcLthtp,GEkkwL7LXo2FF5WvFKpzpQmfNYA1Zh2CoDSwMgVvJhwe,GMGN,False


## Unresolved tokens — 80

The first causal blocker matters: a later migration label can overwrite it.
Seventy-five cases lack sufficient historical seller acquisition/transfer proof.
Five are fixed-transfer conflicts caused by different simulated sender balances.
An unresolved case is not a landed failed sell or a realized loss.

In [9]:
review.display_table('Unresolved')

mint,token,group,reason,first_causal_blocker,execution_failure_reason,entry_SOL,closed_PnL_SOL,failed_fees_SOL,unresolved_cost_SOL,signal_slot,entry_slot,exit_slot,tracked_wallet,developer,gmgn_url,detailed_trace_available
DjWRDKNHZy9x8KwHYzCHAeQXrEVW8j71JpiHhKcNpump,WSB,Unresolved,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,,0.627993300,—,0.000000000,0.627993300,455852379,455852380,<NA>,CWiNFAEk4M1y6SjvnZkmxWDmhqSmnKFcnop1WoDW3S2J,Hkwe8Y9RigKoSTPze4GZU41E7uxZDT67tyCEc59zFTPe,GMGN,False
46JaiTMEt5eAkomaeBgbe7ftjDeaMww5sHU8BqyHpump,DRAUMYR,Unresolved,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,,0.994504960,—,0.000000000,0.994504960,455852491,455852492,<NA>,DxhpC9c4kGkYJR34xUrfFeDx9vwb6yoHUMWrTCZybwtH,HGmHwCi6VShQoTaxJ72e7EfprSaZt4QyTaWYf3jGMK9p,GMGN,False
FT7JCTxVyUJjH6M7zdp4GhpsJtcPK6UQvhFqUqYBpump,BPAYCAT,Unresolved,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,,2.212021888,—,0.000000000,2.212021888,455858840,455858841,<NA>,CWiNFAEk4M1y6SjvnZkmxWDmhqSmnKFcnop1WoDW3S2J,Bnbg4ya7eDfkBMQMXJbbkEWNVS3qoPrQ4CTxcKdGMWBo,GMGN,False
5TwrtsQn6yNJ8ZUjkfnZ7FarJPyUUX447QRKCUd9pump,NORMIE,Unresolved,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,,0.833673640,—,0.000000000,0.833673640,455865712,455865713,<NA>,3Sfme9Wt5S3NPzhc7xZpxwByn3EVYUXgkbHXi9im93An,Hdn2ZbwAGTsixVsM2ZAumhjnDA95a9ZTyndaTWqyXktu,GMGN,False
DHohoH2CjbHQzdVfcPF8zL967ukxUjHkyEFff5vWpump,,Unresolved,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,,1.201924460,—,0.000000000,1.201924460,455879067,455879068,<NA>,3fntCHCB9gBqsZ2WqhajUJ3NqPGPzdj9CrSZe9FiNANS,H2KAvWycyqeHhqkMC1f83Pwju3B2MztLYdXVpNMcDs2V,GMGN,False
BhjFppF6NQiNbMvuLvBTrVbvWmV3FkomNzFe7nrQTQrg,Kellon,Unresolved,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,,2.500301288,—,0.000000000,2.500301288,455885869,455885870,<NA>,CUQqcZZGr8QLjEzMpxUZyiZirBujE2Y9yZDyfCRw1CAB,4vCJzERJRaC7Wkxwzy8hr94QiipJ2w8ZtR6gCG1h2j9u,GMGN,False
8AevP9LKgNoc72pWLMxNPTKsxKQpb8FCgWnUYu9Apump,PADPAD,Unresolved,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,,0.762071359,—,0.000000000,0.762071359,455886585,455886586,<NA>,CWiNFAEk4M1y6SjvnZkmxWDmhqSmnKFcnop1WoDW3S2J,6aoG8GtiScsisQC7unezAvzxAEbokvDb84has4nzKBM9,GMGN,False
CFUAHgH6SFQtRcRKyu2JJGoqQnHu1KxFwhekYaoApump,Fibonacci,Unresolved,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,,4.548362160,—,0.000000000,4.548362160,455887235,455887236,<NA>,23nuEANjatU9atGBJBFcYmoFmae4W3FuFxZhbWRo3FKK,2iV92fS12ud9bTTRVJihcrhKBH48YJQ9t6VCRD9Y4zW6,GMGN,False
Gchzo2KvAKEbJmCB532SA3y54tdK2HifgoMxh93zpump,Campfire,Unresolved,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,,1.581176880,—,0.000000000,1.581176880,455889169,455889170,<NA>,9qz7cWBqYBP2TnGfzMmxX64G4PQip7HtogjPBBnMhWpw,Exbn1cbm6pf9gokxeaEMND4FkuKyHCKbSDdUsWDcnydw,GMGN,False
3afDTMsw7R76MMPRqAQo7rtUrSiG22QaKug5Zpmppump,FS,Unresolved,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,CONDITIONAL_HISTORICAL_SELLER_BALANCE_UNPROVEN,,0.773910900,—,0.000000000,0.773910900,455902513,455902514,<NA>,G9sQPg9Eimkwe72w1xH1tPUc9RFNuK6Ufw5AyWy6j7ob,CJTyd3zhj5PUuTi2bGpt2KgBSh9ACPBdY9wnREYsnBLS,GMGN,False


## Submitted failed entries — 238

These attempts paid the configured execution fee and acquired no token position.
There were zero capital-risk rejections. Failed attempts are shown as × rather
than buy markers.

In [10]:
review.display_table('Entry failed')

mint,token,group,reason,first_causal_blocker,execution_failure_reason,entry_SOL,closed_PnL_SOL,failed_fees_SOL,unresolved_cost_SOL,signal_slot,entry_slot,exit_slot,tracked_wallet,developer,gmgn_url,detailed_trace_available
7bHUu6Lxcy75vuR6EgNuepjwpUUBthDbwVqAode2pump,ARCDOG,Entry failed,RESERVE_DISCONTINUITY,,MINIMUM_OUTPUT_NOT_MET,0.000000000,—,0.000105000,0.000000000,455841299,<NA>,<NA>,DHrKwYTP7aKaiQ8rjBBzbB4Zo2xV1LpbKYE8dsFQZEid,9aHdkTFrbYHB4MZhUU64sUSh1furcLtvwPBGG8Z3WTXz,GMGN,False
2ikMciSZ1nK1X9E8X7U5trw3Tc9jW1Ud7FjrhAxPpump,apelink,Entry failed,MINIMUM_OUTPUT_NOT_MET,,MINIMUM_OUTPUT_NOT_MET,0.000000000,—,0.000105000,0.000000000,455841406,<NA>,<NA>,9HzDhrKAR5Fp6LBmTsNhrUAUC4ZixpYv5Vn1ToA5yTx5,9fJVBF4WaFodQo5BHHHoo9rm6HuJg4P8KRQM23bKHPoa,GMGN,False
HPBYBXbKucX3aE4kQ1WucumFGZmEnVLBuevQAbuupump,muffincat,Entry failed,MINIMUM_OUTPUT_NOT_MET,,MINIMUM_OUTPUT_NOT_MET,0.000000000,—,0.000105000,0.000000000,455842354,<NA>,<NA>,9qz7cWBqYBP2TnGfzMmxX64G4PQip7HtogjPBBnMhWpw,FePSGRavtQty99PFUty5WnaK6M7eibcEEpZRcrmZ6Xi1,GMGN,False
FFnvBUT8ijcLhLML9mfWxQyrLc1ZtA1j8aKGU8bWpump,COMPANY,Entry failed,MINIMUM_OUTPUT_NOT_MET,,MINIMUM_OUTPUT_NOT_MET,0.000000000,—,0.000105000,0.000000000,455842855,<NA>,<NA>,CWiNFAEk4M1y6SjvnZkmxWDmhqSmnKFcnop1WoDW3S2J,CDVHXP2QAKDcwfa1CKvPg1E3hVwrHVw6JDcfXmShXvta,GMGN,False
7S2aR88192Wf3qyHMhjjw1dvUtMg6wKs6zzJkzoNpump,TARDM,Entry failed,MINIMUM_OUTPUT_NOT_MET,,MINIMUM_OUTPUT_NOT_MET,0.000000000,—,0.000105000,0.000000000,455844087,<NA>,<NA>,DTbMwEZ6xfouMAbog188p6FGAk2k9LzJgY3Rd4i5MBRX,ABZdDHYniropNmShEDVNrziLSrTWv7DmUS7htf3EZwx3,GMGN,False
5rRCKzDbNiPLv2SJq5DPbSXMGfngFYZVD1yndvb3pump,CHEER,Entry failed,MINIMUM_OUTPUT_NOT_MET,,MINIMUM_OUTPUT_NOT_MET,0.000000000,—,0.000105000,0.000000000,455844786,<NA>,<NA>,CWiNFAEk4M1y6SjvnZkmxWDmhqSmnKFcnop1WoDW3S2J,3GT13dPNwZ82S5ZB3ntwgx4ueytGTYeJBSyy4XZ83Yqe,GMGN,False
D5DDVQvAkUADg8eiYHmCziY4jdAZRiC8bsEwPPe8pump,PF,Entry failed,MINIMUM_OUTPUT_NOT_MET,,MINIMUM_OUTPUT_NOT_MET,0.000000000,—,0.000105000,0.000000000,455846194,<NA>,<NA>,EeZpAY819RDQavXoJPBC7HB4BTBbfU6rnK7DhyH3G2X6,DdwG8BUEZiqKWSQMtUEjAfZiuRCLTqUJmJtoHFn7GFZU,GMGN,False
DgQbPXHkkuiAAHNQGZuYVKo9wfYp8Xnt3dYdywjXpump,veil,Entry failed,MINIMUM_OUTPUT_NOT_MET,,MINIMUM_OUTPUT_NOT_MET,0.000000000,—,0.000105000,0.000000000,455848769,<NA>,<NA>,EyLQs8oFazJj3ddrDzYMrfSRCsGNfTjM1Z8FkU6aas8N,6TZqJxLLcPQ8AZqhL8XVLdkbQHtFjppBQWimMmgEr6QJ,GMGN,False
B9BDTYZkYdcCLnZFN22FWRd95mu7kVyjuWP9YwFhpump,SCAN,Entry failed,MINIMUM_OUTPUT_NOT_MET,,MINIMUM_OUTPUT_NOT_MET,0.000000000,—,0.000105000,0.000000000,455849854,<NA>,<NA>,6GofQwh2VPeFtDQebwAWfD1ZB1b7L9oB8UA4UUL7QQ3V,6woQGr2TXsPmeijJgLDN7vFX5et2p9szmL955Gc2kenF,GMGN,False
Geq77hxKaY36st5BA7j593nFTHAoivhFtNJ6s4z2qPad,qPad,Entry failed,MINIMUM_OUTPUT_NOT_MET,,MINIMUM_OUTPUT_NOT_MET,0.000000000,—,0.000105000,0.000000000,455851026,<NA>,<NA>,2mqrindMAjJEQPLhroYWyiYPo5h9iAsahfdd4QtsjwdY,5NX7Yv38FKenCEYhoxbWDm9iibWpd6enKFwht6yenf4R,GMGN,False


## Saved token charts

Examples include the largest closed profit, largest closed loss, one unresolved
position, one failed entry and a token with retained detailed evaluations. All
other tokens can be selected above or opened with `review.show(MINT)`.

Detailed per-event strategy traces exist for 78 tokens. Every token still has
recorded order lifecycle events and signal/outcome metadata. Missing evaluations
are labelled in the chart; no decision log is invented.

Chart context starts at launch and ends shortly after exit, or about 6½ minutes
after entry for unresolved positions. It is a bounded view of retained events,
not a complete token history.

In [11]:
examples = [
    review.table[review.table.group=='Profitable'].sort_values('closed_PnL_SOL', ascending=False).iloc[0]['mint'],
    review.table[review.table.group=='Losing'].sort_values('closed_PnL_SOL').iloc[0]['mint'],
    review.table[review.table.group=='Unresolved'].iloc[0]['mint'],
    review.table[review.table.group=='Entry failed'].iloc[0]['mint'],
    review.table[review.table.detailed_trace_available].iloc[0]['mint'],
]
for mint in dict.fromkeys(examples):
    review.show(mint)

,Recorded metadata
status,CLOSED
reason,TARGET
signal_slot,455912387
entry_slot,455912388
exit_signal_slot,455912424
exit_slot,455912425
budget_lamports,4523407920
entry_cash_sol,4.525552200
closed_pnl_sol,1.995450980
failed_execution_cost_sol,0.000000000


**Complete recorded order lifecycle for this token**

,boundary_ordinal,eligible_boundary_ordinal,order_id,phase,record_type,fill_count,reason
0,1958128792006296409,1958128796301262848,0dbfea8de73a2fbdb8d1e9320d87b0a2cec5ca32c39c8d307fb28d8835bda3f5,50,ORDER_ACCEPTED_AND_RESERVED,,
1,1958128796301262848,,0dbfea8de73a2fbdb8d1e9320d87b0a2cec5ca32c39c8d307fb28d8835bda3f5,60,ORDER_FILLED_ATOMICALLY,1,
2,1958128950920086394,1958128955215052800,fd341808535f4575d4a9e1cdf1f8695ce48acbb98a3d110fb62252b63d9d5d01,50,ORDER_ACCEPTED_AND_RESERVED,,
3,1958128955215052800,,fd341808535f4575d4a9e1cdf1f8695ce48acbb98a3d110fb62252b63d9d5d01,60,ORDER_FILLED_ATOMICALLY,1,


Detailed step-by-step strategy evaluations were not retained. Order lifecycle and signal metadata above are recorded evidence.

,Recorded metadata
status,CLOSED
reason,TIMEOUT
signal_slot,456186309
entry_slot,456186310
exit_signal_slot,456187678
exit_slot,456187679
budget_lamports,55290506456
entry_cash_sol,55.292650736
closed_pnl_sol,-31.311714036
failed_execution_cost_sol,0.000000000


...

,Recorded metadata
status,CLOSED
reason,TIMEOUT
signal_slot,455842931
entry_slot,455842932
exit_signal_slot,455844303
exit_slot,455844304
budget_lamports,4671758093
entry_cash_sol,4.673902373
closed_pnl_sol,-0.564938177
failed_execution_cost_sol,0.000000000


**Complete recorded order lifecycle for this token**

,boundary_ordinal,eligible_boundary_ordinal,order_id,phase,record_type,fill_count,reason
0,1957830480757784674,1957830485052751872,7a149afa035c2972efbae364e4f4656703919816c397793948d9b051062072b4,50,ORDER_ACCEPTED_AND_RESERVED,,
1,1957830485052751872,,7a149afa035c2972efbae364e4f4656703919816c397793948d9b051062072b4,60,ORDER_FILLED_ATOMICALLY,1,
2,1957836373452914688,1957836377747881984,8a56a156968de85c2cc659bcbdcbccb883587256343f44e60e18cbc90d48bdcb,50,ORDER_ACCEPTED_AND_RESERVED,,
3,1957836377747881984,,8a56a156968de85c2cc659bcbdcbccb883587256343f44e60e18cbc90d48bdcb,60,ORDER_FILLED_ATOMICALLY,1,


**Retained strategy evaluations — first and last rows**

,mint,symbol,boundary_ordinal,slot,time,action,status,reason,eligible_wallets,wallet,budget_lamports,real_sol,virtual_sol,virtual_tokens,wallet_tokens,...,epsilon,after_virtual_sol,after_real_sol,quoted_tokens,quoted_curve_deposit,min_tokens,side,order_id,reports,execution_fee,pnl_lamports,target_lamports,timeout,gross_curve_sell,synthetic_shortfall
0,78nYgkFzj4uaAgEkkcMjYj8P91k2Wyj2ou5miWBBpump,RAYLS,1957830472167850059,455842929,1791763613,ENTRY_CANDIDATE,NO_SIGNAL,NO_POSITIVE_BREAK_EVEN_WINDOW,1,8dx4RHfjJhskX2aY2GJ1aJPLGRpJtW13FjRcknVdExSo,0,4938271603,34938271603,921339222665955,151660777334045,...,,,,,,,,,,,,,,,
1,78nYgkFzj4uaAgEkkcMjYj8P91k2Wyj2ou5miWBBpump,RAYLS,1957830472167850059,455842929,1791763613,ENTRY_NO_WINDOW,NO_SIGNAL,NO_POSITIVE_BREAK_EVEN_WINDOW,1,,,,34938271603,921339222665955,,...,,,,,,,,,,,,,,,
2,78nYgkFzj4uaAgEkkcMjYj8P91k2Wyj2ou5miWBBpump,RAYLS,1957830472167850060,455842929,1791763613,ENTRY_CANDIDATE,NO_SIGNAL,NO_POSITIVE_BREAK_EVEN_WINDOW,1,8dx4RHfjJhskX2aY2GJ1aJPLGRpJtW13FjRcknVdExSo,0,5827160491,35827160491,898480358444437,151660777334045,...,,,,,,,,,,,,,,,
3,78nYgkFzj4uaAgEkkcMjYj8P91k2Wyj2ou5miWBBpump,RAYLS,1957830472167850060,455842929,1791763613,ENTRY_NO_WINDOW,NO_SIGNAL,NO_POSITIVE_BREAK_EVEN_WINDOW,1,,,,35827160491,898480358444437,,...,,,,,,,,,,,,,,,
4,78nYgkFzj4uaAgEkkcMjYj8P91k2Wyj2ou5miWBBpump,RAYLS,1957830472167850082,455842929,1791763613,ENTRY_CANDIDATE,NO_SIGNAL,NO_POSITIVE_BREAK_EVEN_WINDOW,2,8dx4RHfjJhskX2aY2GJ1aJPLGRpJtW13FjRcknVdExSo,0,7308641971,37308641971,862802779738918,151660777334045,...,,,,,,,,,,,,,,,
5,78nYgkFzj4uaAgEkkcMjYj8P91k2Wyj2ou5miWBBpump,RAYLS,1957830472167850082,455842929,1791763613,ENTRY_CANDIDATE,NO_SIGNAL,NO_POSITIVE_BREAK_EVEN_WINDOW,2,DxhpC9c4kGkYJR34xUrfFeDx9vwb6yoHUMWrTCZybwtH,0,7308641971,37308641971,862802779738918,35677578705519,...,,,,,,,,,,,,,,,
6,78nYgkFzj4uaAgEkkcMjYj8P91k2Wyj2ou5miWBBpump,RAYLS,1957836347683110912,455844297,1791763911,EXIT_EVALUATION,OPEN,HOLDING,2,DxhpC9c4kGkYJR34xUrfFeDx9vwb6yoHUMWrTCZybwtH,,4388216789,34388216789,936076453280771,0,...,,,,,,,,,,,-564938177,233593155,False,4159017637,0
7,78nYgkFzj4uaAgEkkcMjYj8P91k2Wyj2ou5miWBBpump,RAYLS,1957836351978078208,455844298,1791763911,EXIT_EVALUATION,OPEN,HOLDING,2,DxhpC9c4kGkYJR34xUrfFeDx9vwb6yoHUMWrTCZybwtH,,4388216789,34388216789,936076453280771,0,...,,,,,,,,,,,-564938177,233593155,False,4159017637,0
8,78nYgkFzj4uaAgEkkcMjYj8P91k2Wyj2ou5miWBBpump,RAYLS,1957836356273045504,455844299,1791763911,EXIT_EVALUATION,OPEN,HOLDING,2,DxhpC9c4kGkYJR34xUrfFeDx9vwb6yoHUMWrTCZybwtH,,4388216789,34388216789,936076453280771,0,...,,,,,,,,,,,-564938177,233593155,False,4159017637,0
9,78nYgkFzj4uaAgEkkcMjYj8P91k2Wyj2ou5miWBBpump,RAYLS,1957836360568012800,455844300,1791763912,EXIT_EVALUATION,OPEN,HOLDING,2,DxhpC9c4kGkYJR34xUrfFeDx9vwb6yoHUMWrTCZybwtH,,4388216789,34388216789,936076453280771,0,...,,,,,,,,,,,-564938177,233593155,False,4159017637,0


## Open a particular token

Replace `TOKEN` with a mint or paste a full GMGN token URL. The code keeps the
address and discards a URL query string.

In [12]:
TOKEN = None  # e.g. 'https://gmgn.ai/sol/token/<mint>'
if TOKEN:
    mint = TOKEN.rstrip('/').split('/')[-1].split('?')[0]
    assert mint in review.positions[LABEL], 'This token is not among the 887 signals'
    review.show(mint)

## CSVs and verification

CSV exports contain all positions by outcome, exact fill markers, realized PnL
events, cash changes and open-cost events. The notebook verifies native file
hashes, asset conservation in every ledger transaction, all order identities,
and reconciliation to the saved PnL and exposure totals.

The original cache and native result remain unchanged. This is an evidence review
of one simulated day, not a new backtest or independent validation period.

In [13]:
csv_folder = review.export()
display(pd.Series({
    'source_input_sha256': review.summary['source_input_sha256'],
    'native_source_sha256': review.summary['implementation']['engine_source_sha256'],
    'orders_matched_by_exact_native_hash': len(review.orders),
    'actual_buy_markers': sum(f['side']=='BUY' for f in review.fill_markers),
    'actual_sell_markers': sum(f['side']=='SELL' for f in review.fill_markers),
    'realized_event_total_SOL': sum(r['pnl_lamports'] for r in review.pnl_events)/1e9,
    'open_cost_event_total_SOL': sum(r['delta'] for r in review.exposure_events)/1e9,
    'csv_folder': str(csv_folder),
}, name='Verification').to_frame())

,Verification
source_input_sha256,dd62a8ac8fb2f3981b50af94e8b9af378b760e9dd62735f7dd490092e188bdd3
native_source_sha256,f93a8de0259cfd5de2fea152df072ad80a09aa72adb5bedbfebbcc2766df62f0
orders_matched_by_exact_native_hash,1475
actual_buy_markers,649
actual_sell_markers,569
realized_event_total_SOL,-109.602394
open_cost_event_total_SOL,152.31915
csv_folder,./var/dni...
